# Stage 6, Project 1: Spam Classifier

Full project template: problem → data → cleaning → features → baseline → models → evaluation → error analysis → improvements.

**Problem:** classify SMS messages as spam or ham (not spam), using only the message text.

**Dataset:** SMS Spam Collection (5,574 labeled messages, loaded directly from a public URL below — no manual download needed).

## 1. Load and first look

Guiding questions:
- What's the class balance between spam and ham? (This matters a lot for which metrics you'll trust later.)
- Look at a few raw examples of each class — what surface-level patterns do you notice in spam messages (length, punctuation, capitalization, specific words)?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

# TODO: load with pd.read_csv, note it's tab-separated (sep='\t') with no header row
# name the columns something like ['label', 'message']




In [6]:
df = pd.read_csv(
    url,
    sep='\t',
    header=None,
    names=['label', 'message']
)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
# TODO: class balance (value_counts), a few example messages from each class


example_messages = df.value_counts()
print(example_messages)


label  message                                                                                                                                                                            
ham    Sorry, I'll call later                                                                                                                                                                 30
       I cant pick the phone right now. Pls send a message                                                                                                                                    12
       Ok...                                                                                                                                                                                  10
spam   Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £5000 prize!                                     4
ham    Wen ur lovable bcums angry wid u, 

## 2. Text cleaning

Unlike Titanic, there's no missing-value/outlier work here — the "cleaning" for text data is normalization: lowercasing, removing punctuation, maybe removing extra whitespace.

Guiding question: should you remove stopwords ("the", "is", "a"...) for spam detection specifically? Think about whether function words might actually carry signal here (e.g., do spam messages use different sentence structure/punctuation density than real texts) before assuming stopword removal always helps.


Answer:
- I think we should not remove stopwords for spam detection, certain uses of word and language may be helpful in indicating and carrying a signal if a message is spam or not.

In [10]:
import re

def clean_text(text):
    # TODO: lowercase, strip punctuation (re.sub), collapse extra whitespace
    text = text.lower()
    
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# TODO: apply to the message column, store as a new 'clean_message' column
df['clean_message'] = df['message'].apply(clean_text)


## 3. Feature engineering: text → numbers

Models can't read raw text — you need to convert messages into numeric feature vectors. Two standard approaches:

- **Bag-of-words (`CountVectorizer`)**: counts how many times each word appears
- **TF-IDF (`TfidfVectorizer`)**: like bag-of-words, but downweights words that appear in almost every message (less informative) and upweights words that are rare-but-distinctive

Guiding question: for spam detection specifically, why might TF-IDF's downweighting of common words be especially useful compared to raw counts?

Try both in this notebook and compare — don't just pick one blindly.

Answer:
- For spam detection specifically, TD-IDF's downweighting of common words that appear frequently across both spam and legitimate messages, while giving more weight to rare but distinctive words. This allows the model to focus on the more distinct words that are informative in distinguishing spam vs non spam.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer


# TODO: split BEFORE vectorizing (same leakage rule as always — fit the vectorizer on train only)
# X = clean_message column, y = label column (map to 0/1: ham=0, spam=1)

# X = input text
X = df['clean_message']

# y = target
# ham = 0, spam = 1
y = df['label'].map({'ham': 0, 'spam': 1})

# 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split the 30% temporary set into:
# 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)




In [13]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# TODO: fit CountVectorizer on X_train only, transform X_train and X_val
# TODO: separately, fit TfidfVectorizer on X_train only, transform X_train and X_val
# keep both versions around — you'll compare models on each

count_vectorizer = CountVectorizer()
count_vectorizer.fit(X_train)

X_train_count_vec = count_vectorizer.transform(X_train)
X_val_count_vec = count_vectorizer.transform(X_val)
X_test_count_vec = count_vectorizer.transform(X_test)



# -------------------- - -- - - - - -
tfid_vectorizer = TfidfVectorizer()
tfid_vectorizer.fit(X_train)

X_train_tfid_vec = tfid_vectorizer.transform(X_train)
X_val_tfid_vec = tfid_vectorizer.transform(X_val)
X_test_tfid__vec = tfid_vectorizer.transform(X_test)





## 4. Baseline model

Before any real model, what would a "dumb" baseline score? Given the class imbalance you found in Section 1, what accuracy would you get by always predicting the majority class (ham)? Compute this — it's the number every real model needs to beat by a meaningful margin, not just barely.

In [19]:
# TODO: majority-class baseline accuracy

label_count = df.value_counts("label")

ham_count = label_count["ham"]
spam_count = label_count["spam"]

baseline_score = ham_count / (ham_count + spam_count)

print(baseline_score) # This means that .86 of emails are not spam whic is a 'dumb' baseline score given our data.

0.8659368269921034


## 5. Train and compare models

Train both Naive Bayes and Logistic Regression, on both the CountVectorizer and TF-IDF feature sets (4 combinations total). This comparison *is* the actual lesson of this project — you're not just picking a winner, you're seeing how algorithm choice and feature choice interact.

Guiding question before running anything: Naive Bayes assumes features (words) are conditionally independent given the class. Is that assumption more or less violated in text data than in, say, the Titanic features? Why might Naive Bayes still work well for text despite this?


Answer:
- I think that given the assumption that words are conditionally independent give the class, it is more violated in text data since texts sewn together help classify if a email is spam or not. Naive bayes still work well for text despite this becuase individiual words can still provide strong signals for classifcation. Naive Bayes is also computationally efficient and works well with high-dimensional, sparse text data.

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# --------------------------------------------
# Naive Bayes + CountVectorizer
# --------------------------------------------

MultinomialNB_model = MultinomialNB()

MultinomialNB_model.fit(X_train_count_vec, y_train)

y_val_pred_count_NB = MultinomialNB_model.predict(X_val_count_vec)


# --------------------------------------------
# Naive Bayes + TF-IDF
# --------------------------------------------

MultinomialNB_model.fit(X_train_tfid_vec, y_train)

y_val_pred_tfid_NB = MultinomialNB_model.predict(X_val_tfid_vec)


# --------------------------------------------
# Logistic Regression + CountVectorizer
# --------------------------------------------

LogisticRegression_model = LogisticRegression()

LogisticRegression_model.fit(X_train_count_vec, y_train)

y_val_pred_count_LR = LogisticRegression_model.predict(X_val_count_vec)


# --------------------------------------------
# Logistic Regression + TF-IDF
# --------------------------------------------

LogisticRegression_model.fit(X_train_tfid_vec, y_train)

y_val_pred_tfid_LR = LogisticRegression_model.predict(X_val_tfid_vec)

## 6. Evaluate all four combinations

This is spam detection — given the class imbalance, which metric(s) from Stage 5 should you lead with instead of accuracy? Think specifically about which error type (false positive: real message flagged as spam, vs false negative: spam message let through) is more costly to a real user, and let that guide whether you care more about precision or recall here.

Build a small comparison table (rows = 4 model/feature combos, columns = precision, recall, F1) rather than eyeballing scattered print statements.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# TODO: evaluate all 4 model/feature combinations, build a comparison table
results = []

# NB + Count
results.append({
    'Model': 'Naive Bayes',
    'Features': 'CountVectorizer',
    'Precision': precision_score(y_val, y_val_pred_count_NB),
    'Recall': recall_score(y_val, y_val_pred_count_NB),
    'F1': f1_score(y_val, y_val_pred_count_NB)
})

# NB + TF-IDF
results.append({
    'Model': 'Naive Bayes',
    'Features': 'TF-IDF',
    'Precision': precision_score(y_val, y_val_pred_tfid_NB),
    'Recall': recall_score(y_val, y_val_pred_tfid_NB),
    'F1': f1_score(y_val, y_val_pred_tfid_NB)
})

# LR + Count
results.append({
    'Model': 'Logistic Regression',
    'Features': 'CountVectorizer',
    'Precision': precision_score(y_val, y_val_pred_count_LR),
    'Recall': recall_score(y_val, y_val_pred_count_LR),
    'F1': f1_score(y_val, y_val_pred_count_LR)
})

# LR + TF-IDF
results.append({
    'Model': 'Logistic Regression',
    'Features': 'TF-IDF',
    'Precision': precision_score(y_val, y_val_pred_tfid_LR),
    'Recall': recall_score(y_val, y_val_pred_tfid_LR),
    'F1': f1_score(y_val, y_val_pred_tfid_LR)
})

results_df = pd.DataFrame(results)
results_df



,Model,Features,Precision,Recall,F1
0,Naive Bayes,CountVectorizer,0.970000,0.866071,0.915094
1,Naive Bayes,TF-IDF,1.000000,0.616071,0.762431
2,Logistic Regression,CountVectorizer,0.989796,0.866071,0.923810
3,Logistic Regression,TF-IDF,0.987342,0.696429,0.816754


## 7. Error analysis

Pick your best-performing combination. Pull out the actual messages your model got wrong (both false positives and false negatives) and read them.

Guiding questions:
- Do the false positives (real messages flagged as spam) share anything in common — lots of numbers, urgent-sounding language, all-caps?
- Do the false negatives (spam that got through) look unusually subtle, or short, or different from typical spam patterns?
- This is the step most people skip, and it's the one that actually tells you *how* to improve the model — not just that it's imperfect, but specifically where and why.

Answer:
- The model appears to struggle with unconventional SMS vocabulary and subtle spam messages that don't contain obvious spam keywords.
- The false positives appear to be normal messages that happen to contain 'spam' like influenced words.
- The false negatives appear to be misplaced because of the misuse vocabuarly and strugged with SMS langauge. This suggests that the model has difficulty distinguishing between classes when it does not contain common words and patterns it learned during training.

In [29]:
# TODO: pull out and print the misclassified messages for your best model, split into FP and FN
error_df = pd.DataFrame({
    'message': X_val,
    'actual': y_val,
    'predicted': y_val_pred_count_LR
})

false_positives = error_df[
    (error_df['actual'] == 0) &
    (error_df['predicted'] == 1)
]

false_negatives = error_df[
    (error_df['actual'] == 1) &
    (error_df['predicted'] == 0)
]

print("FALSE POSITIVES")
print(false_positives['message'].to_string(index=False))

print("\nFALSE NEGATIVES")
print(false_negatives['message'].to_string(index=False))

FALSE POSITIVES
misplaced your number and was sending texts to ...

FALSE NEGATIVES
network operator the service is free for t cs v...
freemsg hey u i just got 1 of these videopic fo...
adult 18 content your video will be with you sh...
download as many ringtones as u like no restric...
hi ya babe x u 4goten bout me scammers getting ...
not heard from u4 a while call me now am here a...
more people are dogging in your area now call 0...
     3 you have received your mobile content enjoy
                   freemsgfav xmas tonesreply real
bought one ringtone and now getting texts costi...
                                ringtoneking 84484
hi if ur lookin 4 saucy daytime fun wiv busty m...
email alertfrom jeri stewartsize 2kbsubject low...
            rct thnq adrian for u text rgds vatian
romcapspam everyone around should be responding...


## 8. Improvements to attempt

Pick at least one and try it, based on what your error analysis revealed:
- N-grams instead of single words (`ngram_range=(1,2)` in the vectorizer) — captures short phrases like "free entry" as a single feature
- Adding message length or digit-count as an extra numeric feature alongside the text features
- Trying `class_weight='balanced'` in LogisticRegression to see if it shifts the precision/recall tradeoff favorably
- Tuning the decision threshold (like you did in Stage 5) instead of using the default 0.5

In [ ]:
# TODO: implement and evaluate at least one improvement, compare against your Section 6 best result

# N-games:


#Model
tfid_vectorizer_ngram = TfidfVectorizer(
    ngram_range=(1, 2)
)

#Fit
tfid_vectorizer_ngram.fit(X_train)

#Transform
X_train_ngram = tfid_vectorizer_ngram.transform(X_train)
X_val_ngram = tfid_vectorizer_ngram.transform(X_val)
X_test_ngram = tfid_vectorizer_ngram.transform(X_test)

#Train model
lr_ngram = LogisticRegression()
lr_ngram.fit(X_train_ngram, y_train)
y_val_pred_ngram = lr_ngram.predict(X_val_ngram)

precision_ngram = precision_score(y_val, y_val_pred_ngram)
recall_ngram = recall_score(y_val, y_val_pred_ngram)
f1_ngram = f1_score(y_val, y_val_pred_ngram)

print("TF-IDF + N-grams")
print("Precision:", precision_ngram)
print("Recall:", recall_ngram)
print("F1:", f1_ngram)



#Logistic Regression	
# TF-IDF	
# Precision: 0.987342	
# Recall:    0.696429	
# F1:        0.816754


TF-IDF + N-grams
Precision: 1.0
Recall: 0.6517857142857143
F1: 0.7891891891891892


Anayalsis:

When we use TF-IDF with n-grams, the model is provided with more context because it can consider combinations of two words in addition to individual words. This allows the model to recognize phrases such as "free entry" or "call now" as features, rather than treating each word independently. As a result, the model may be better able to distinguish spam from legitimate messages and make more accurate predictions.

## Wrap-up

Write a short summary:
- Which model/feature combination won, and by which metric did you judge "winning"?
- What did error analysis reveal that a metrics table alone wouldn't have shown you?
- Did your improvement attempt actually help? If not, what does that tell you?


Answer: 
- The Logistic regression and CountVectorizer combination won, it scored the highest in precision, recall and in f1 score. The metrics table only showed me the number of TN,FP,etc, but during error analysis I was able to see the descrepancy and determine why the model is making these false postivies and true negatives. I was able to see certain phrases, or words that could have contributed to these errors. Yes my improvement attempt actually helped. Using n-grams, I learned that giving the model additional features can provide more context for making predictions. By including two-word combinations instead of only individual words, the model can recognize phrases and patterns that may be more informative for identifying spam. In this experiment, the additional contextual information helped the model make more accurate predictions.